# 02 — Dataset Preparation (VQAv2 subset)

**Goal:** download a small, configurable subset of VQAv2 and verify the
preprocessing pipeline in `src/dataset.py` produces correctly shaped tensors.

We use a **subset** (see `configs/config.yaml` -> `dataset.train_subset_size`)
because the full VQAv2 train split (~443K questions) is not practical to even
just download/preprocess repeatedly on a Colab session. The subset size is a
config knob you can raise later.

In [ ]:
!pip install -q datasets transformers torch pyyaml

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.train import load_config
cfg = load_config(os.path.join(PROJECT_ROOT, "configs", "config.yaml"))
cfg["dataset"]

In [ ]:
from src.dataset import load_vqa_subset, VQADataset

train_split = load_vqa_subset(
    cfg["dataset"]["hf_dataset_name"], "train",
    cfg["dataset"]["train_subset_size"], cfg["dataset"]["seed"],
)
val_split = load_vqa_subset(
    cfg["dataset"]["hf_dataset_name"], "validation",
    cfg["dataset"]["val_subset_size"], cfg["dataset"]["seed"],
)
print("Train subset size:", len(train_split))
print("Val subset size:", len(val_split))

**Common errors & fixes:**\n- First download can take a few minutes and several hundred MB — normal.\n- `ConnectionError` → HF Hub hiccup, just retry the cell.\n- If you hit Colab disk limits, lower `train_subset_size` / `val_subset_size` in `configs/config.yaml`.

In [ ]:
train_ds = VQADataset(train_split)
sample = train_ds[0]
print("Question:", sample["question"])
print("Majority answer:", sample["answer"])
sample["image"]

In [ ]:
from functools import partial
from torch.utils.data import DataLoader
from src.model import load_model
from src.dataset import collate_fn

processor, model, device, _ = load_model(model_name=cfg["model"]["name"])
collate = partial(
    collate_fn, processor=processor,
    max_question_length=cfg["model"]["max_question_length"],
    max_answer_length=cfg["model"]["max_answer_length"],
)
loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=collate)
batch = next(iter(loader))
for k, v in batch.items():
    print(k, v.shape)

**Expected output:** tensor shapes like `pixel_values torch.Size([4, 3, 384, 384])`, `input_ids torch.Size([4, 32])`, `labels torch.Size([4, 8])` (exact numbers depend on your config). This confirms the preprocessing pipeline is correct before we start training.\n\n**Next notebook:** `03_finetuning.ipynb`